### Project Ariadne Delta Logprob Datasets

To measure the correlation between the ID-deltas and the OOD-deltas, we've got to create both datasets
from the samples we've collected in the previous step. 

For the OOD-Data, we've collected ground truth samples from the Qwen2.5-7B basemodel and augmented them using Qwen3-8B. For the ID-Data, we've collected rollouts for each checkpoint $i \in \{0, 10, 20, ..., 80\}$ and distilled answers for which Qwen3-8B decided they contain a calculation error. This gives us pairs $(x_{\text{gt}}, x_{\text{aug}})$ for the OOD-data and pairs $(x_{\text{gt}}, x_{\text{error}})$ for the ID-samples.

The question is, whether $\rho = \text{corr}(\Delta_{\text{ID}}, \Delta_{\text{OOD}}) > 0$ 

In [1]:
import os
import numpy as np
import pandas as pd

#### OOD Answers

Let's load in the generated answers, group them to the original ground truths and then create the ood-deltas dataset.

In [29]:
with open('/u/rfechner/data/ariadne/ood-outputs-per-checkpoint.parquet', 'rb') as file:
    df_ood_gen = pd.read_parquet(file)

In [32]:
df_ood_gen.iloc[1]['responses'][0]

'<think>\nOkay, let\'s see. The user wants me to inject an arithmetic or algebraic error into the student\'s answer, but only if it\'s reasonable. The original answer is correct, so I need to find a place where a small mistake could be made without changing the overall structure.\n\nLooking at the solution steps: The correct steps are setting up the equation 124 + 2x = 180, subtracting 124 from both sides to get 2x = 56, then dividing by 2 to get x = 28. \n\nWhere could an error occur here? Maybe in the subtraction step. Instead of 180 - 124, someone might mistakenly subtract 124 from 180 as 56, but that\'s correct. Wait, maybe in the division step. If they thought 56 divided by 2 is 28, which is right. Hmm. Alternatively, maybe they added instead of subtracted? Like 124 + 2x = 180, then 2x = 180 + 124? That would be a mistake. Let me check that. If they did 180 + 124 = 304, then 304 divided by 2 is 152. But that\'s a big error. But the original answer is 28, so maybe a smaller error. 

In [33]:
def filter_parsed_ood_samples(df : pd.DataFrame) -> pd.DataFrame:
    """
        We'll apply a simple heuristic to decide whether an answer is a good augmentation:
        1) It must contain a '####' marker signaling the start of the answer.
        2) it must contain "\\boxed{" within the last 100 characters
    """
    def mapper(responses : list[str]) -> bool | str:
        r = responses[0] # only a single response per row
        try:
            i = r.rindex('####')
        except ValueError:
            return False
        if "boxed{" not in r[i:]:
            return False
        return r[i:].removeprefix('####').lstrip()

    df['parsed_response'] = df['responses'].apply(mapper)
    df = df[df['parsed_response'].astype(bool)]
    return df

df_ood_gen_parsed = filter_parsed_ood_samples(df_ood_gen.copy())

In [34]:
def input_from_prompt(prompt : str) -> str:
    qprefix = "\nQuestion:\n"
    qindex = prompt.rindex(qprefix)
    aprefix = "\nSolution:\n"
    aindex = prompt.rindex(aprefix)
    question = prompt[qindex:aindex].removeprefix(qprefix)
    return question

def ground_truth_from_prompt(prompt : str) -> str:
    aprefix = "\nSolution:\n"
    aindex = prompt.rindex(aprefix)
    gt = prompt[aindex:].removeprefix(aprefix).removesuffix('\n')
    return gt

In [35]:
a = df_ood_gen_parsed.iloc[0]['prompt'][1]['content']
input_from_prompt(a), ground_truth_from_prompt(a)

('$n$ fair 6-sided dice are simultaneously rolled. The probability that exactly two of them show a number other than 1 is $\\frac{25}{216}$. Find $n$.',
 'To solve this problem, we need to find the number of dice ($n$) such that the probability of exactly two dice showing a number other than 1 is $\\frac{25}{216}$.\n\n1. **Determine the probability of a single die showing a number other than 1:**\n   - There are 5 outcomes out of 6 that are not 1 (i.e., 2, 3, 4, 5, 6).\n   - So, the probability is $\\frac{5}{6}$.\n\n2. **Determine the probability of a single die showing a 1:**\n   - There is 1 outcome out of 6 that is 1.\n   - So, the probability is $\\frac{1}{6}$.\n\n3. **Use the binomial probability formula:**\n   - We want exactly 2 dice to show a number other than 1, and the rest to show 1.\n   - The probability of exactly 2 successes (dice showing a number other than 1) in $n$ trials (dice rolls) is given by the binomial probability formula:\n     \\[\n     P(X = 2) = \\binom{n}{2

In [43]:
rows = []
for i, row in df_ood_gen_parsed.iterrows():
    prompt = row['prompt'][1]['content']
    inp, gt = input_from_prompt(prompt), ground_truth_from_prompt(prompt)
    step = row['step']
    behavior = row['parsed_response']
    correct = {
        'prompt' : [{'role' : 'user', 'content' : inp}, 
                    {'role' : 'assistant', 'content' : gt}],
        'behaviour_type' : 'anchor',
        'step' : step,
        "uid" : i
    }
    incorrect = {
        'prompt' : [{'role' : 'user', 'content' : inp}, 
                    {'role' : 'assistant', 'content' : behavior}],
        'behaviour_type' : 'calculation_error',
        'step' : step,
        "uid" : i
    }
    rows.append(correct); rows.append(incorrect)

In [44]:
ood_dataframe = pd.DataFrame(rows)
ood_dataframe.head(1)

,prompt,behaviour_type,step,uid
0,"[{'role': 'user', 'content': '$n$ fair 6-sided...",anchor,0,111687


In [45]:
# at the time of writing this, I have to make the number of lines in the deltas dataframe I'm passing divisible by 8.
# It is what it is.

len(ood_dataframe)

1914

In [46]:
ood_dataframe.groupby('step').apply(len)

/tmp/ipykernel_4141/3642159773.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ood_dataframe.groupby('step').apply(len)


step
0     268
10    240
20    222
30    220
40    192
50    196
60    166
70    212
80    198
dtype: int64

In [47]:
drop_last = len(ood_dataframe) % 8
ood_dataframe = ood_dataframe[:-drop_last]
assert len(ood_dataframe) % 8 == 0

In [48]:
path = "/u/rfechner/data/ariadne"
os.makedirs(path, exist_ok=True)

with open(os.path.join(path, 'ood-deltas-per-checkpoint.jsonl'), 'w') as file:
    ood_dataframe.to_json(path_or_buf=file, lines=True, orient='records')